# 04 - External-predictor tabular models

This notebook introduces **external-predictor tabular models**: classical
regression/tree models trained on engineered daily feature rows built from
calendar information and external satellite/reanalysis/meteorological
predictors, rather than from the target series' own recent history.

It covers:

1. what external-predictor tabular models are and how they differ from the
   Model 0 baselines in `03_baselines.ipynb`;
2. why temporal awareness has to be hand-engineered into these features
   (lags, rolling summaries, anomalies, availability flags) since a plain
   tabular row has no built-in notion of sequence;
3. how the curated external predictor table
   (`data_public/chlorophyll/chlorophyll_predictor_features_curated.csv`)
   is structured and how to load it;
4. a real finding from this project: external predictors alone did not
   clearly beat simple interpolation in this low-data, local setting;
5. a small, runnable example training a tree model on artificial gaps.

## 1. What is an external-predictor tabular model?

A tabular model treats each calendar day as one row of a feature table and
predicts a single target value (here, daily mean chlorophyll-a) from the
other columns in that row -- the same setup as any standard regression or
classification problem. The "external-predictor" qualifier means the
feature columns are restricted to information that does not depend on the
target series' own recent observed values: calendar position, satellite
sea-surface temperature, a satellite chlorophyll proxy, wind, and
upwelling-related variables from a nearby meteorological station and
reanalysis products.

This is deliberately different from a *gap-edge* model (see
`05_gap_edge_residual_models.ipynb`), which is allowed to look at the
target's own value immediately before and after a gap. External-predictor
models are safe to use on **any** gap, including very long ones or gaps
near the edge of the record, because they never require a recent target
observation to exist.

## 2. Why temporal awareness must be engineered by hand

A single row of a tabular model has no built-in concept of "yesterday" or
"a rolling average of the last week" -- each row is treated independently
by most regression/tree algorithms. If the underlying process has memory
(today's chlorophyll is correlated with yesterday's wind, or with a
multi-day upwelling trend), that memory has to be exposed explicitly as
extra columns:

- **Lags** -- the value of a predictor N days earlier (e.g.
  `wind_u_ms_lag3`, `mur_sst_degC` shifted by a fixed offset).
- **Rolling summaries** -- a moving average or sum over a trailing window
  (e.g. `plv_solar_roll7d_wm2`, `cmems_upwelling_cumul14d_ms_d`).
- **Anomalies** -- a predictor's deviation from its typical seasonal value
  (e.g. `mur_sst_anom_doy_degC`, `chl_anom_log10_monthly`), which separates
  an unusually warm/cool day from the normal seasonal cycle.
- **Availability flags** -- a boolean column (e.g. `wind_available`,
  `chl_cons_available`) recording whether the underlying source actually
  had data that day, since satellite products have their own gaps
  (cloud cover, swath coverage) independent of the in-situ sensor's gaps.

This is real, manual development effort: each lag/rolling/anomaly variant
is a deliberate design decision, not something a plain tabular model
infers automatically. The curated feature table in this repository already
has many of these variants pre-computed; building an equivalent table from
scratch for a new sensor or site requires re-doing this work (see
`docs/methodology/model_families.md` and
`notebooks/09_adapting_the_workflow_to_a_new_sensor.ipynb`).

## 3. Loading the curated external-predictor feature table

`chlorophyll_predictor_features_curated.csv` has one row per calendar day
(3,988 rows) and 126 columns. Column families include:

- calendar: `season`, `day_of_year`, `month`, `year`, `doy_sin`, `doy_cos`
- satellite chlorophyll proxy: `chl_cons_log10`, `chl_perm_log10`, and
  their lag/roll/anomaly/patchiness variants
- sea-surface temperature: `mur_sst_degC`, `ostia_sst_degC`,
  `sst_primary_degC`, and SST gradient/frontal features
- wind: `wind_u_ms`, `wind_v_ms`, `wind_spd_ms` (CMEMS reanalysis) and
  `plv_wind_*` (nearby meteorological station)
- meteorological forcing: `plv_temp_degC`, `plv_pressure_hPa`,
  `plv_humid_pct`, `plv_precip_daily_mm`, `plv_solar_wm2`
- upwelling indices: `plv_upwelling_ms`, `cmems_upwelling_ms`, and their
  cumulative/relaxation-index variants

See `docs/data_dictionary.md` for the full column-by-column listing.

In [ ]:
import pandas as pd

features = pd.read_csv(
    "../data_public/chlorophyll/chlorophyll_predictor_features_curated.csv",
    parse_dates=["date"],
)
print(features.shape)
features[[
    "date", "season", "chl_cons_log10", "mur_sst_degC",
    "wind_spd_ms", "plv_upwelling_ms",
]].head()

## 4. Why external predictors alone did not clearly beat interpolation here

A real finding from this project, not a hypothetical caveat: in this
benchmark's artificial-gap validation, external-predictor tabular models
(trained only on the feature families above, with no access to the
target's own recent history) did **not** show a clear, statistically
significant improvement over linear interpolation across most gap
lengths. Two probabilistic sequence models (a Gaussian process and a
state-space/Kalman model, see `docs/methodology/model_families.md`) and a
zero-shot foundation model (TS-ICL, see
`notebooks/06_tsicl_zero_shot_imputation.ipynb`) performed competitively
or better. The validated comparison numbers are in
`results_public/chlorophyll/chlorophyll_benchmark_summary.csv` and
`results_public/chlorophyll/chlorophyll_artificial_gap_scores.csv`.

Plausible reasons, in this local, relatively low-data setting (roughly a
decade of daily data at a single station):

- the record is short enough that a tabular model has limited examples to
  learn from, especially for less common conditions (long gaps, event
  days);
- external predictors capture broad physical forcing (temperature, wind,
  upwelling) but do not directly observe the target variable's own
  short-term persistence, which linear interpolation exploits by
  construction over short gaps;
- chlorophyll-a at this site is noisy and event-driven (see
  `docs/methodology/event_limitation.md`), and external predictors alone
  do not fully explain that variability.

This is a genuine negative/mixed result worth keeping in view: more
feature engineering and a more complex model family are not guaranteed to
beat a much simpler baseline, and that comparison should always be checked
against validation-grade evidence (see `docs/evidence_hierarchy.md`)
rather than assumed.

## 5. A small, runnable example: training a tree model on artificial gaps

This is a minimal, illustrative example -- it is not the production
pipeline behind `chlorophyll_reconstruction_engineered_hybrid.csv`, just a
small `RandomForestRegressor` trained on a handful of external-predictor
columns, evaluated against a sample of artificial gaps from the canonical
validation pool (`chlorophyll_validation_gaps.csv`). It uses only
non-target-history features, consistent with section 1 above.

The example deliberately keeps the feature list small (under 10 columns)
for clarity; a real model would use a larger, carefully chosen subset (or
all 126 columns with regularization), selected via an explicit ablation
plan rather than added all at once (see
`docs/methodology/model_families.md`).

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor

target = pd.read_csv(
    "../data_public/chlorophyll/chlorophyll_daily_target.csv", parse_dates=["date"]
)
gaps = pd.read_csv(
    "../data_public/chlorophyll/chlorophyll_validation_gaps.csv",
    parse_dates=["start_date", "end_date"],
)

# Build a daily table: target (log10 chlorophyll) + a small external feature set.
daily = target[["date", "target_eligible_default", "chl_mean"]].merge(
    features[[
        "date", "doy_sin", "doy_cos", "mur_sst_degC", "wind_spd_ms",
        "plv_upwelling_ms", "chl_cons_log10",
    ]],
    on="date", how="left",
)
daily["chl_log10"] = np.log10(daily["chl_mean"].clip(lower=0.01))

feature_cols = ["doy_sin", "doy_cos", "mur_sst_degC", "wind_spd_ms",
                 "plv_upwelling_ms", "chl_cons_log10"]

# Train on all eligible days that are NOT inside any artificial gap window,
# so the model never sees the gap's hidden days during training.
gap_dates = set()
for _, g in gaps.iterrows():
    gap_dates.update(pd.date_range(g["start_date"], g["end_date"]))

train_mask = (
    daily["target_eligible_default"]
    & daily["chl_log10"].notna()
    & daily[feature_cols].notna().all(axis=1)
    & ~daily["date"].isin(gap_dates)
)
train = daily[train_mask]

model = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=0)
model.fit(train[feature_cols], train["chl_log10"])

# Evaluate on a sample of artificial gaps with usable features and known truth.
sample_gaps = gaps.sample(n=min(30, len(gaps)), random_state=0)
errors = []
for _, g in sample_gaps.iterrows():
    hidden_dates = pd.date_range(g["start_date"], g["end_date"])
    rows = daily[daily["date"].isin(hidden_dates) & daily[feature_cols].notna().all(axis=1)]
    if rows.empty:
        continue
    pred_log10 = model.predict(rows[feature_cols])
    pred_chl = 10 ** pred_log10
    true_mean = g["target_mean_true"]
    errors.append(abs(pred_chl.mean() - true_mean))

print(f"Sampled {len(errors)} gaps with usable features.")
print(f"Mean absolute error (gap-mean chlorophyll): {np.mean(errors):.4f}")
print(
    "Compare against the validation-grade method comparison in "
    "results_public/chlorophyll/chlorophyll_benchmark_summary.csv -- "
    "this small illustrative model is not expected to match the tuned "
    "engineered hybrid or TS-ICL pipelines."
)

## 6. Takeaways

- External-predictor tabular models are useful because they generalize to
  any gap length and any position in the record, with no dependence on
  recent target observations.
- Building a good feature table is real, manual work: lags, rolling
  windows, anomalies, and availability flags all have to be designed
  deliberately, ideally tracked in a feature registry with an explicit
  ablation plan rather than added all at once.
- In this project's low-data, single-station setting, external predictors
  alone did not clearly outperform linear interpolation under
  validation-grade testing -- see
  `results_public/chlorophyll/chlorophyll_benchmark_summary.csv` for the
  numbers, and `docs/evidence_hierarchy.md` before drawing conclusions
  from any other table in this repository.
- See `05_gap_edge_residual_models.ipynb` for a complementary model family
  that additionally uses gap-edge information, and
  `06_tsicl_zero_shot_imputation.ipynb` for the leading method in this
  benchmark under artificial-gap validation.